# Pick preictal ingestion targets

This notebook finds sample-level preictal overlap candidates and builds ready-to-run ingestion commands.

In [ ]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from epilepsiae_sql_dataloader.utils import _normalize_pgurl

In [ ]:
PGURL = os.environ.get('PGURL')
if not PGURL:
    raise RuntimeError('Set PGURL in your environment before running this notebook')

ENGINE_STR = _normalize_pgurl(PGURL)
engine = create_engine(ENGINE_STR)
print('Using:', ENGINE_STR)

In [ ]:
QUERY = text("""
with candidates as (
  select
    z.pat_id,
    sm.id as sample_id,
    sm.start_ts,
    sm.duration_in_sec,
    z.onset,
    z.offset,
    extract(epoch from (z.onset - sm.start_ts))::int as sec_from_start,
    greatest(
      0,
      extract(
        epoch from
        least(sm.start_ts + (sm.duration_in_sec || ' seconds')::interval, z.onset)
        - greatest(sm.start_ts, z.onset - interval '3600 seconds')
      )
    )::int as preictal_overlap_s
  from seizures z
  join samples sm on sm.pat_id = z.pat_id
  where sm.start_ts < z.onset
    and (sm.start_ts + (sm.duration_in_sec || ' seconds')::interval) > (z.onset - interval '3600 seconds')
)
select *
from candidates
where preictal_overlap_s > 0
order by preictal_overlap_s desc, pat_id, sample_id
limit 200
""")

with engine.connect() as conn:
    candidates = pd.read_sql(QUERY, conn)

print('rows:', len(candidates))
candidates.head(20)

In [ ]:
margin_s = 120
default_max_seconds = 1200

targets = candidates.copy()
targets['recommended_start_seconds'] = np.maximum(
    0,
    targets['sec_from_start'].astype(int) - 3600 - margin_s,
).astype(int)
targets['recommended_max_seconds'] = np.minimum(
    default_max_seconds,
    np.maximum(0, targets['duration_in_sec'].astype(int) - targets['recommended_start_seconds'])
).astype(int)

targets = targets[targets['recommended_max_seconds'] > 0].copy()
targets['ingest_cmd'] = targets.apply(
    lambda r: (
        f"python -m epilepsiae_sql_dataloader.RelationalRigging.PushBinaryToSql \"
        f"--dir '$DATA_ROOT/inv' --sample-ids {int(r.sample_id)} --start-seconds {int(r.recommended_start_seconds)} \"
        f"--max-seconds {int(r.recommended_max_seconds)} --verbose"
    ),
    axis=1,
)
targets['export_cmd'] = targets.apply(
    lambda r: (
        f"python -m epilepsiae_sql_dataloader.DataDinghy.ExportChunksToParquet \"
        f"--pgurl '$PGURL' --out '/tmp/preictal_pat_{int(r.pat_id)}.parquet' --patients {int(r.pat_id)} \"
        f"--states 0,2 --near-seizure preictal --data-types 0 --window-seconds 60 --stride-seconds 10 --verbose"
    ),
    axis=1,
)

cols = [
    'pat_id', 'sample_id', 'start_ts', 'onset', 'preictal_overlap_s',
    'recommended_start_seconds', 'recommended_max_seconds',
    'ingest_cmd', 'export_cmd',
]
targets[cols].head(20)

In [ ]:
for i, row in targets.head(10).iterrows():
    print('-' * 100)
    print(f"pat_id={int(row.pat_id)} sample_id={int(row.sample_id)} preictal_overlap_s={int(row.preictal_overlap_s)}")
    print(row.ingest_cmd)
    print(row.export_cmd)